**Data Collection & Preparation**

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
weather_url = "https://archive-api.open-meteo.com/v1/archive"

weather_params = {
    "latitude": "55.476,56.1393,57.0488,56.1629,55.4038,54.9138",
    "longitude": "8.459,8.9738,9.9217,10.2039,10.4024,9.7926",
    "start_date": "2024-12-31",
    "end_date": "2025-12-31",
    "hourly": "temperature_2m,wind_speed_100m,shortwave_radiation",
    "wind_speed_unit": "ms",
    "timezone": "UTC"
}

weather_response = requests.get(weather_url, params=weather_params, timeout=60)
weather_response.raise_for_status()
weather_data = weather_response.json()

weather_frames = []

for data in weather_data:
    location_df = pd.DataFrame(data["hourly"])
    location_df["time"] = pd.to_datetime(location_df["time"], utc=True)
    weather_frames.append(location_df)

weather_df = pd.concat(weather_frames, ignore_index=True)
weather_df = weather_df.groupby("time", as_index=False).mean(numeric_only=True)

display(weather_df.head())

In [ ]:
print("Table dimensions:", weather_df.shape)
print("Average wind speed:", round(weather_df["wind_speed_100m"].mean(), 2))
print("Maximum wind speed:", round(weather_df["wind_speed_100m"].max(), 2))
print("Minimum wind speed:", round(weather_df["wind_speed_100m"].min(), 2))
print("Average temperature:", round(weather_df["temperature_2m"].mean(), 2))

wind_weekly = weather_df.set_index("time")["wind_speed_100m"].resample("W").mean()

plt.figure(figsize=(12, 5))
plt.plot(wind_weekly.index, wind_weekly, label="Average Wind Speed — DK1", linewidth=2)

plt.title("Weekly Average Wind Speed — DK1 (2025)")
plt.xlabel("Date")
plt.ylabel("Wind Speed (m/s)")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
generation_url = "https://api.energidataservice.dk/dataset/GenerationProdTypeExchange"

generation_params = {
    "start": "2025-01-01T00:00",
    "end": "2026-01-01T00:00",
    "filter": '{"PriceArea":["DK1"]}',
    "limit": 0,
    "sort": "TimeUTC asc"
}

generation_response = requests.get(generation_url, params=generation_params, timeout=60)
generation_response.raise_for_status()
generation_data = generation_response.json()

generation_clean = pd.DataFrame(generation_data["records"])[["TimeUTC", "PriceArea", "Version", "OffshoreWindPower", "OnshoreWindPower"]].copy()

generation_clean["TimeUTC"] = pd.to_datetime(generation_clean["TimeUTC"], utc=True)
generation_clean["WindGeneration"] = generation_clean["OffshoreWindPower"] + generation_clean["OnshoreWindPower"]

generation_clean = generation_clean.sort_values("TimeUTC").reset_index(drop=True)

display(generation_clean.head())

In [ ]:
power_df = pd.merge(weather_df, generation_clean, left_on="time", right_on="TimeUTC", how="inner", validate="one_to_one")
power_df = power_df.drop(columns="TimeUTC")

print("Final dataset:", power_df.shape)
print("Missing values:", power_df.isna().sum().sum())

display(power_df.head())

**Exploratory Data Analysis**

In [ ]:
correlation = power_df["wind_speed_100m"].corr(power_df["WindGeneration"])
print("Wind Speed vs Wind Generation Correlation:", round(correlation, 3))

plt.figure(figsize=(10, 6))
plt.scatter(power_df["wind_speed_100m"], power_df["WindGeneration"], alpha=0.25, s=10)

plt.title("Wind Speed vs Wind Generation — DK1 (2025)")
plt.xlabel("Average Wind Speed at 100m (m/s)")
plt.ylabel("Total Wind Generation")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
wind_profile = power_df[["wind_speed_100m", "WindGeneration"]].copy()
wind_profile["wind_speed_rounded"] = wind_profile["wind_speed_100m"].round()

wind_profile = wind_profile.groupby("wind_speed_rounded")["WindGeneration"].mean()

plt.figure(figsize=(10, 6))
plt.plot(wind_profile.index, wind_profile.values, marker="o")

plt.title("Average Wind Generation by Wind Speed — DK1")
plt.xlabel("Wind Speed at 100m (m/s)")
plt.ylabel("Average Wind Generation")
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
monthly_data = power_df.set_index("time").tz_convert("Europe/Copenhagen")

generation_monthly = monthly_data["WindGeneration"].resample("MS").mean()
wind_monthly = monthly_data["wind_speed_100m"].resample("MS").mean()

plt.figure(figsize=(12, 5))
plt.plot(generation_monthly.index, generation_monthly, marker="o", linewidth=2)
plt.xticks(generation_monthly.index, generation_monthly.index.strftime("%b"))
plt.title("Monthly Average Wind Generation — DK1 (2025)")
plt.xlabel("Month")
plt.ylabel("Average Wind Generation")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(wind_monthly.index, wind_monthly, marker="o", linewidth=2)
plt.xticks(wind_monthly.index, wind_monthly.index.strftime("%b"))
plt.title("Monthly Average Wind Speed — DK1 (2025)")
plt.xlabel("Month")
plt.ylabel("Average Wind Speed (m/s)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
weather_corr = power_df[["wind_speed_100m", "temperature_2m", "shortwave_radiation", "WindGeneration"]].corr()["WindGeneration"].drop("WindGeneration").sort_values(ascending=False)

print(weather_corr.round(3).to_string())

plt.figure(figsize=(10, 5))
plt.scatter(power_df["temperature_2m"], power_df["WindGeneration"], alpha=0.3, s=10)
plt.title("Temperature vs Wind Generation — DK1")
plt.xlabel("Temperature at 2m (°C)")
plt.ylabel("Wind Generation")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.scatter(power_df["shortwave_radiation"], power_df["WindGeneration"], alpha=0.3, s=10)
plt.title("Solar Radiation vs Wind Generation — DK1")
plt.xlabel("Shortwave Radiation (W/m²)")
plt.ylabel("Wind Generation")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Model Development & Validation**

In [ ]:
model_df = power_df.sort_values("time").reset_index(drop=True)

X = model_df[["wind_speed_100m"]]
y = model_df["WindGeneration"]

validation_split_index = int(len(model_df) * 0.64)
split_index = int(len(model_df) * 0.80)

X_train = X.iloc[:validation_split_index]
X_val = X.iloc[validation_split_index:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:validation_split_index]
y_val = y.iloc[validation_split_index:split_index]
y_test = y.iloc[split_index:]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

In [ ]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_val)

baseline_mae = mean_absolute_error(y_val, y_pred_baseline)
baseline_rmse = mean_squared_error(y_val, y_pred_baseline) ** 0.5
baseline_r2 = r2_score(y_val, y_pred_baseline)

model_comparison = pd.DataFrame({
    "Model": ["Baseline Linear Regression"],
    "MAE": [baseline_mae],
    "RMSE": [baseline_rmse],
    "R²": [baseline_r2]
})

print("Intercept:", round(baseline_model.intercept_, 2))
print("Wind speed coefficient:", round(baseline_model.coef_[0], 2))

display(model_comparison.round(3))

In [ ]:
X_multi = model_df[["wind_speed_100m", "temperature_2m", "shortwave_radiation"]]

X_multi_train = X_multi.iloc[:validation_split_index]
X_multi_val = X_multi.iloc[validation_split_index:split_index]

multi_model = LinearRegression()
multi_model.fit(X_multi_train, y_train)

y_pred_multi = multi_model.predict(X_multi_val)

multi_mae = mean_absolute_error(y_val, y_pred_multi)
multi_rmse = mean_squared_error(y_val, y_pred_multi) ** 0.5
multi_r2 = r2_score(y_val, y_pred_multi)

model_comparison = model_comparison[model_comparison["Model"] != "Multiple Linear Regression"]
model_comparison.loc[len(model_comparison)] = ["Multiple Linear Regression", multi_mae, multi_rmse, multi_r2]

display(model_comparison.round(3))

In [ ]:
poly_features = PolynomialFeatures(degree=2, include_bias=False)

X_poly_train = poly_features.fit_transform(X_multi_train)
X_poly_val = poly_features.transform(X_multi_val)

poly_model = LinearRegression()
poly_model.fit(X_poly_train, y_train)

y_pred_poly = poly_model.predict(X_poly_val)

poly_mae = mean_absolute_error(y_val, y_pred_poly)
poly_rmse = mean_squared_error(y_val, y_pred_poly) ** 0.5
poly_r2 = r2_score(y_val, y_pred_poly)

model_comparison.loc[model_comparison["Model"] == "Polynomial Regression", ["MAE", "RMSE", "R²"]] = [poly_mae, poly_rmse, poly_r2]

display(model_comparison.round(3))

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_multi_train, y_train)

y_pred_rf = rf_model.predict(X_multi_val)

rf_mae = mean_absolute_error(y_val, y_pred_rf)
rf_rmse = mean_squared_error(y_val, y_pred_rf) ** 0.5
rf_r2 = r2_score(y_val, y_pred_rf)

model_comparison = model_comparison[model_comparison["Model"] != "Random Forest"]
model_comparison.loc[len(model_comparison)] = ["Random Forest", rf_mae, rf_rmse, rf_r2]

display(model_comparison.round(3))

In [ ]:
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_multi_train, y_train)

y_pred_gb = gb_model.predict(X_multi_val)

gb_mae = mean_absolute_error(y_val, y_pred_gb)
gb_rmse = mean_squared_error(y_val, y_pred_gb) ** 0.5
gb_r2 = r2_score(y_val, y_pred_gb)

model_comparison = model_comparison[model_comparison["Model"] != "Gradient Boosting"]
model_comparison.loc[len(model_comparison)] = ["Gradient Boosting", gb_mae, gb_rmse, gb_r2]

display(model_comparison.round(3))

In [ ]:
gb_configs = [
    ("Default", 100, 0.1, 3),
    ("200 Trees", 200, 0.05, 3),
    ("300 Trees", 300, 0.05, 3),
    ("Lower Learning Rate", 300, 0.03, 3),
    ("Shallower Trees", 200, 0.05, 2)
]

gb_results = []

for name, trees, rate, depth in gb_configs:
    model = GradientBoostingRegressor(n_estimators=trees, learning_rate=rate, max_depth=depth, random_state=42)
    model.fit(X_multi_train, y_train)
    predictions = model.predict(X_multi_val)

    gb_results.append([name, mean_absolute_error(y_val, predictions), mean_squared_error(y_val, predictions) ** 0.5, r2_score(y_val, predictions)])

gb_tuning_comparison = pd.DataFrame(gb_results, columns=["Model", "MAE", "RMSE", "R²"])
display(gb_tuning_comparison.round(3).sort_values("MAE"))

**Final Model Evaluation**

In [ ]:
final_gb_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=42)
final_gb_model.fit(X_multi.iloc[:split_index], y.iloc[:split_index])

y_pred_final_gb = final_gb_model.predict(X_multi.iloc[split_index:])

baseline_test_model = LinearRegression()
baseline_test_model.fit(X.iloc[:split_index], y.iloc[:split_index])
y_pred_baseline_test = baseline_test_model.predict(X.iloc[split_index:])

test_comparison = pd.DataFrame({
    "Model": ["Baseline Linear Regression", "Tuned Gradient Boosting"],
    "MAE": [mean_absolute_error(y_test, y_pred_baseline_test), mean_absolute_error(y_test, y_pred_final_gb)],
    "RMSE": [mean_squared_error(y_test, y_pred_baseline_test) ** 0.5, mean_squared_error(y_test, y_pred_final_gb) ** 0.5],
    "R²": [r2_score(y_test, y_pred_baseline_test), r2_score(y_test, y_pred_final_gb)]
})

display(test_comparison.round(3))

In [ ]:
final_comparison_df = model_df.iloc[split_index:][["time"]].copy()

final_comparison_df["Actual Generation"] = y_test.values
final_comparison_df["Predicted Generation"] = y_pred_final_gb
final_comparison_df["Residual"] = final_comparison_df["Actual Generation"] - final_comparison_df["Predicted Generation"]

plt.figure(figsize=(13, 6))
plt.plot(final_comparison_df["time"], final_comparison_df["Actual Generation"], label="Actual Generation")
plt.plot(final_comparison_df["time"], final_comparison_df["Predicted Generation"], label="Predicted Generation")
plt.title("Actual vs Predicted Wind Generation — DK1")
plt.xlabel("Date")
plt.ylabel("Wind Generation")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
plt.plot(final_comparison_df["time"], final_comparison_df["Residual"])
plt.axhline(y=0, linestyle="--")
plt.title("Residuals Over Time — DK1")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(final_comparison_df["Residual"], bins=30, edgecolor="black", alpha=0.7)
plt.axvline(x=0, linestyle="--", label="Zero Error")
plt.title("Residual Distribution — DK1")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Day-Ahead Wind Power Forecast**

In [ ]:
forecast_url = "https://previous-runs-api.open-meteo.com/v1/forecast"

forecast_params = {
    "latitude": "55.476,56.1393,57.0488,56.1629,55.4038,54.9138",
    "longitude": "8.459,8.9738,9.9217,10.2039,10.4024,9.7926",
    "start_date": "2024-12-31",
    "end_date": "2025-12-31",
    "hourly": "temperature_2m_previous_day1,wind_speed_100m_previous_day1,shortwave_radiation_previous_day1",
    "wind_speed_unit": "ms",
    "timezone": "UTC"
}

forecast_response = requests.get(forecast_url, params=forecast_params, timeout=60)
forecast_response.raise_for_status()
forecast_data = forecast_response.json()

forecast_frames = []

for data in forecast_data:
    location_df = pd.DataFrame(data["hourly"])
    location_df["time"] = pd.to_datetime(location_df["time"], utc=True)
    forecast_frames.append(location_df)

weather_forecast_df = (
    pd.concat(forecast_frames, ignore_index=True)
    .groupby("time", as_index=False)
    .mean(numeric_only=True)
    .rename(columns={
        "temperature_2m_previous_day1": "temperature_2m_forecast",
        "wind_speed_100m_previous_day1": "wind_speed_100m_forecast",
        "shortwave_radiation_previous_day1": "shortwave_radiation_forecast"
    })
)

print("Weather Forecast Dimensions:", weather_forecast_df.shape)
display(weather_forecast_df.head())


In [ ]:
forecast_power_df = pd.merge(weather_forecast_df, generation_clean, left_on="time", right_on="TimeUTC", how="inner", validate="one_to_one")
forecast_power_df = forecast_power_df.drop(columns="TimeUTC")

print("Forecast Power Dataset:", forecast_power_df.shape)
print("Missing values:", forecast_power_df.isna().sum().sum())

display(forecast_power_df.head())

In [ ]:
X_forecast = forecast_power_df[["wind_speed_100m_forecast", "temperature_2m_forecast", "shortwave_radiation_forecast"]]
y_forecast = forecast_power_df["WindGeneration"]

X_forecast_train = X_forecast.iloc[:split_index]
X_forecast_test = X_forecast.iloc[split_index:]

y_forecast_train = y_forecast.iloc[:split_index]
y_forecast_test = y_forecast.iloc[split_index:]

print("Training:", len(X_forecast_train))
print("Testing:", len(X_forecast_test))

In [ ]:
forecast_gb_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=42)
forecast_gb_model.fit(X_forecast_train, y_forecast_train)

y_pred_forecast = forecast_gb_model.predict(X_forecast_test)

forecast_comparison = pd.DataFrame({
    "Model": ["Observed Weather GB", "Day-Ahead Weather GB"],
    "MAE": [mean_absolute_error(y_test, y_pred_final_gb), mean_absolute_error(y_forecast_test, y_pred_forecast)],
    "RMSE": [mean_squared_error(y_test, y_pred_final_gb) ** 0.5, mean_squared_error(y_forecast_test, y_pred_forecast) ** 0.5],
    "R²": [r2_score(y_test, y_pred_final_gb), r2_score(y_forecast_test, y_pred_forecast)]
})

display(forecast_comparison.round(3))

In [ ]:
forecast_comparison_df = forecast_power_df.iloc[split_index:][["time"]].copy()

forecast_comparison_df["Actual Generation"] = y_forecast_test.values
forecast_comparison_df["Predicted Generation"] = y_pred_forecast
forecast_comparison_df["Residual"] = forecast_comparison_df["Actual Generation"] - forecast_comparison_df["Predicted Generation"]

plt.figure(figsize=(13, 6))
plt.plot(forecast_comparison_df["time"], forecast_comparison_df["Actual Generation"], label="Actual Generation")
plt.plot(forecast_comparison_df["time"], forecast_comparison_df["Predicted Generation"], label="Day-Ahead Prediction")
plt.title("Actual vs Predicted Wind Generation — Day-Ahead")
plt.xlabel("Date")
plt.ylabel("Wind Generation")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 5))
plt.plot(forecast_comparison_df["time"], forecast_comparison_df["Residual"])
plt.axhline(y=0, linestyle="--")
plt.title("Residuals Over Time — Day-Ahead")
plt.xlabel("Date")
plt.ylabel("Residual")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(forecast_comparison_df["Residual"], bins=30, edgecolor="black", alpha=0.7)
plt.axvline(x=0, linestyle="--", label="Zero Error")
plt.title("Residual Distribution — Day-Ahead")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Walk-Forward Backtesting**

In [ ]:
months_to_test = ["2025-10", "2025-11", "2025-12"]
forecast_months = forecast_power_df["time"].dt.tz_convert("Europe/Copenhagen").dt.strftime("%Y-%m")
test_start = forecast_power_df["time"].iloc[split_index]

walk_forward_results = []
walk_forward_predictions = []

for month in months_to_test:
    month_start = pd.Timestamp(month, tz="Europe/Copenhagen").tz_convert("UTC")
    test_mask = (forecast_months == month) & (forecast_power_df["time"] >= test_start)
    train_mask = forecast_power_df["time"] < max(month_start, test_start)

    wf_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=42)
    wf_model.fit(X_forecast.loc[train_mask], y_forecast.loc[train_mask])
    actual = y_forecast.loc[test_mask]
    predictions = wf_model.predict(X_forecast.loc[test_mask])

    walk_forward_results.append([month, train_mask.sum(), test_mask.sum(), mean_absolute_error(actual, predictions), mean_squared_error(actual, predictions) ** 0.5, r2_score(actual, predictions)])
    walk_forward_predictions.append(pd.DataFrame({"Actual Generation": actual.values, "Predicted Generation": predictions}))

walk_forward_results_df = pd.DataFrame(walk_forward_results, columns=["Month", "Training Observations", "Testing Observations", "MAE", "RMSE", "R²"])
walk_forward_predictions_df = pd.concat(walk_forward_predictions, ignore_index=True)

display(walk_forward_results_df.round(3))

actual_all = walk_forward_predictions_df["Actual Generation"]
predicted_all = walk_forward_predictions_df["Predicted Generation"]

print("Overall MAE:", round(mean_absolute_error(actual_all, predicted_all), 3))
print("Overall RMSE:", round(mean_squared_error(actual_all, predicted_all) ** 0.5, 3))
print("Overall R²:", round(r2_score(actual_all, predicted_all), 3))

**Conclusion**

1. Project Overview

This project developed a machine learning model to forecast wind power generation in Denmark's DK1 bidding zone using historical weather data and weather forecasts.

2. Key Results

The Gradient Boosting model achieved the best performance, with the following Day-Ahead test results:

*   MAE	385.110
*   RMSE	509.651
*   R²	0.848

The Walk-Forward Backtest produced an overall R² of 0.850, with an MAE of 383.105 and an RMSE of 506.773.

3. Conclusion

The results show that weather forecasts can provide useful information for predicting wind power generation 24 hours ahead. The model performed well on the evaluated period, although prediction errors increased compared with using observed weather data.

4. Limitations

The model uses only three weather variables and one year of historical data. Factors such as turbine availability, power plant outages, curtailment and grid constraints are not included. Further testing across additional years is needed to assess its reliability.

5. Future Improvements

Future work could include additional operational data, longer historical periods and more detailed forecast simulations aligned with actual Day-Ahead market publication times.